# Session 03 — Draw a helpful face-framing guide

Today’s new skill: use drawings on a webcam frame to tell a person what to do.

OpenCV finds a face-shaped region and returns a box. We use that box to draw a target and give one clear response: READY, ADJUST, or FACE NOT FOUND.

This detects face position only. It does not identify who the person is.


## Choose your webcam route

- On your own computer: use the local continuous live-webcam loop first.
- In Colab: use the browser-based continuous live preview.

Both routes use the same face detector, alignment rule, and drawing code.


## Teacher setup — do this once

This prepares OpenCV for the Colab runtime. It is not part of the student task. Run it once, restart the runtime when it tells you to, then students begin at **1. Choose the rule you will test**.


In [ ]:
# Run this setup cell once. Do not continue to the next cell yet.
from pathlib import Path
import subprocess
import sys

ready_file = Path('/content/session3_opencv_ready')

if not ready_file.exists():
    # OpenCV packages all use the same cv2 name. Keep exactly one.
    for package in ['cv2', 'opencv-python', 'opencv-python-headless', 'opencv-contrib-python']:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'uninstall', '-y', package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False
        )

    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        '--quiet', '--no-cache-dir', 'opencv-python-headless==4.13.0.92'
    ])
    ready_file.touch()
    print('OpenCV is installed. Choose Runtime > Restart session, then choose Runtime > Run all.')
else:
    import cv2
    import numpy as np
    print('OpenCV:', cv2.__version__)


## 1. Choose the rule you will test

Start with the suggested value. Later, change only READY_TOLERANCE and observe how the decision changes.


In [ ]:
# These two imports give our code its image and drawing tools.
import cv2

# Size of the blue target box in the middle of the frame.
TARGET_WIDTH = 360
TARGET_HEIGHT = 240
MIN_FACE_SIZE = 80

# Your first edit: how close must the face centre be to the target centre?
READY_TOLERANCE = 24

# OpenCV includes this pretrained face detector. It finds faces; it does not recognise identities.
face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

if face_detector.empty():
    raise RuntimeError('The face-detector file could not load. Restart the runtime and run from the top.')


## 2. Find one face box

The detector returns rectangles: x, y, width, height. We keep the largest visible face.


In [ ]:
# Turn the image into grey shades so the detector can look for a face pattern.
def find_face_box(frame):
    grey = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    boxes = face_detector.detectMultiScale(
        grey, scaleFactor=1.1, minNeighbors=5,
        minSize=(MIN_FACE_SIZE, MIN_FACE_SIZE)
    )

    # If more than one face appears, use the largest box.
    return max(boxes, key=lambda box: box[2] * box[3]) if len(boxes) else None


## 3. Draw the guide and choose a status

This helper draws the target, face box, centre points, and a clear message.


In [ ]:
def draw_alignment_guide(frame):
    # Keep the original frame unchanged; draw on a copy instead.
    output = frame.copy()
    frame_height, frame_width = output.shape[:2]
    # Calculate a target box centred in the frame.
    left = (frame_width - TARGET_WIDTH) // 2
    top = (frame_height - TARGET_HEIGHT) // 2
    right = left + TARGET_WIDTH
    bottom = top + TARGET_HEIGHT
    target_centre = (left + TARGET_WIDTH // 2, top + TARGET_HEIGHT // 2)

    blue, cyan, orange, green = (21,138,194), (255,210,0), (246,138,69), (68,180,92)
    # Draw the target box, centre line, and centre cross.
    cv2.rectangle(output, (left, top), (right, bottom), cyan, 3)
    cv2.line(output, (frame_width // 2, top), (frame_width // 2, bottom), cyan, 2)
    cv2.drawMarker(output, target_centre, cyan, cv2.MARKER_CROSS, 20, 2)

    # Ask the earlier helper whether a face is visible in this frame.
    face_box = find_face_box(frame)
    status = 'FACE NOT FOUND — move into view'
    status_colour = orange
    if face_box is not None:
        x, y, width, height = face_box
        face_centre = (x + width // 2, y + height // 2)
        # Measure how far the face centre is from the target centre.
        distance = ((face_centre[0] - target_centre[0]) ** 2 +
                    (face_centre[1] - target_centre[1]) ** 2) ** 0.5
        ready = left <= face_centre[0] <= right and top <= face_centre[1] <= bottom and distance <= READY_TOLERANCE
        cv2.rectangle(output, (x, y), (x + width, y + height), blue, 3)
        status_colour = green if ready else orange
        cv2.circle(output, face_centre, 8, status_colour, -1)
        cv2.line(output, target_centre, face_centre, status_colour, 2)
        status = 'READY — hold position' if ready else 'ADJUST — centre your face'

    # Add a readable status bar and show the chosen tolerance.
    cv2.rectangle(output, (14, 14), (frame_width - 14, 60), (18, 25, 34), -1)
    cv2.putText(output, status, (26, 45), cv2.FONT_HERSHEY_SIMPLEX, .65, status_colour, 2)
    cv2.putText(output, f'tolerance: {READY_TOLERANCE}px', (18, frame_height - 18),
                cv2.FONT_HERSHEY_SIMPLEX, .6, (255,255,255), 2)
    return output, status


## 4A. Local route — continuous live webcam

Run this on your own computer, in a local Python or Jupyter environment. It reads, checks, and redraws **every frame**. Press **q** in the camera window to stop.


In [ ]:
# Ask OpenCV for frames from the computer webcam.
camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print('No local webcam was found. In Colab, use the live-preview cell below.')
else:
    while True:
        # Read the next webcam frame, then redraw the guide on that new frame.
        ok, frame = camera.read()
        if not ok:
            break
        result, status = draw_alignment_guide(frame)
        cv2.imshow('Face framing guide — press q to exit', result)

        # Stop only when the learner presses q.
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the camera for other apps and close the preview window.
    camera.release()
    cv2.destroyAllWindows()


## 4B. Colab route — continuous live preview

Run this in Colab. The browser sends small webcam frames to Python repeatedly. Python finds the face, draws the guide, and sends the updated frame back. It is a live preview, although it updates more slowly than the local route because each frame travels between the browser and Python. Press **STOP PREVIEW** when you finish.


In [ ]:
# NumPy turns browser image bytes into an image OpenCV can read.
import numpy as np
from base64 import b64decode, b64encode
from IPython.display import JSON, Javascript, display
from google.colab import output

# This function receives one webcam frame from the browser.
def process_live_frame(image_data):
    # Remove the web-image header, then turn the remaining text back into image bytes.
    image_bytes = b64decode(image_data.split(',', 1)[1])
    frame = cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)

    # Draw the same guide and face box used by the local live route.
    result, status = draw_alignment_guide(frame)

    # Turn the finished OpenCV image back into text the browser can display.
    _, encoded_image = cv2.imencode('.jpg', result)
    result_data = 'data:image/jpeg;base64,' + b64encode(encoded_image).decode('utf-8')
    # JSON is the response format the Colab browser bridge reads.
    return JSON({'image': result_data, 'status': status})

# Give the browser a short name it can use to call the Python function above.
output.register_callback('session3.process_frame', process_live_frame)

display(Javascript(r'''
async function startSession3LivePreview() {
  // Remove an earlier preview if this cell is run again.
  document.getElementById('session3-live-preview')?.remove();

  const panel = document.createElement('div');
  const message = document.createElement('p');
  const start = document.createElement('button');
  const stop = document.createElement('button');
  const video = document.createElement('video');
  const rawFrame = document.createElement('canvas');
  const preview = document.createElement('canvas');
  let running = false;
  let stream;

  panel.id = 'session3-live-preview';
  panel.style.cssText = 'max-width:640px;padding:12px;border:2px solid #10151a;';
  start.textContent = 'START LIVE PREVIEW';
  stop.textContent = 'STOP PREVIEW';
  stop.disabled = true;
  start.style.cssText = stop.style.cssText = 'padding:10px 16px;font-weight:bold;margin:8px 8px 8px 0;';
  video.style.display = 'none';
  preview.style.cssText = 'display:block;width:100%;border:2px solid #10151a;box-sizing:border-box;';
  message.textContent = 'Click START LIVE PREVIEW, then allow camera access in the browser prompt.';
  panel.append(message, start, stop, video, preview);
  document.body.appendChild(panel);

  async function showNextFrame() {
    if (!running) return;

    // Copy the newest browser-camera frame into a small canvas.
    rawFrame.width = video.videoWidth;
    rawFrame.height = video.videoHeight;
    rawFrame.getContext('2d').drawImage(video, 0, 0);

    try {
      // Ask Python to find the face and draw the interface on this frame.
      const response = await google.colab.kernel.invokeFunction(
        'session3.process_frame',
        [rawFrame.toDataURL('image/jpeg', 0.65)],
        {}
      );
      const result = response.data['application/json'];
      const image = new Image();
      image.onload = () => {
        preview.width = image.width;
        preview.height = image.height;
        preview.getContext('2d').drawImage(image, 0, 0);
      };
      image.src = result.image;
      message.textContent = result.status + ' — live preview is updating.';
    } catch (error) {
      console.error(error);
      message.textContent = 'Preview stopped: ' + (error.message || error);
      running = false;
      start.disabled = false;
      stop.disabled = true;
      return;
    }

    // A short pause keeps the browser and Python responsive for the learner.
    setTimeout(showNextFrame, 80);
  }

  start.onclick = async () => {
    try {
      stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream;
      await video.play();
      running = true;
      start.disabled = true;
      stop.disabled = false;
      message.textContent = 'Camera is live. Python is checking new frames.';
      showNextFrame();
    } catch (error) {
      message.textContent = 'Camera permission was denied. Allow the camera in browser site settings, then run this cell again.';
    }
  };

  stop.onclick = () => {
    running = false;
    stream?.getTracks().forEach(track => track.stop());
    start.disabled = false;
    stop.disabled = true;
    message.textContent = 'Preview stopped.';
  };
}
startSession3LivePreview();
'''))


## 5. Record evidence

1. With your face centred, what status appears?
2. With it off-centre, what status appears?
3. When it is out of view, what status appears?
4. After changing READY_TOLERANCE, does the guide approve too early, too late, or at a useful point?

Exit evidence: write one rule you changed, one visible result, and one case where a user could still be confused.
